# 🎬 PeakCut Studio — Google Colab Edition (Reliable GPU Pipeline)

Turn long-form videos (authorized YouTube URLs or uploaded MP4/MOV/MKV files) into **vertical 9:16 shorts with animated karaoke subtitles**, powered by **Faster-Whisper**, **smart reframing**, and **local Ollama AI (Gemma)** — with **zero paid API keys required**.

### Quick Start (Free T4 GPU)
1. In Google Colab, select **Runtime → Change runtime type → T4 GPU** (CPU fallback is also supported, though slower).
2. Run **Cells 1 → 6** in order to verify your runtime, install dependencies cleanly, connect Google Drive, and verify local AI.
3. For the **Web Dashboard**: run **Cell 7** (start & verify FastAPI backend) → **Cell 8** (open authenticated Cloudflare tunnel or private Colab proxy).
4. For **Direct Notebook Processing**: run **Cell 9** with a YouTube URL or uploaded video file, then preview/download in **Cell 10**.
5. If anything misbehaves, run **Cell 11 ("Diagnose PeakCut")** for an automated 14-point health report and log excerpts.

> **Persistence & Safety:** FFmpeg stages temporary work on fast local SSD (`/content/PeakCut_Work`), validates every MP4 with `ffprobe`, and copies finished clips + subtitles + metadata to `/content/drive/MyDrive/PeakCut_Shorts/` (or `/content/PeakCut_Output/` when Drive is not mounted). Every cell is safe to rerun.

## Cell 1 — Runtime & GPU Diagnostics

In [ ]:
# @title 1. Inspect Runtime, GPU, CUDA, RAM & Disk
import os
import platform
import shutil
import subprocess
import sys

print("=" * 70)
print("🖥️  PeakCut Studio — Cell 1: Runtime & Hardware Diagnostics")
print("=" * 70)
print(f"Python       : {platform.python_version()} ({sys.executable})")
print(f"OS / Arch    : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"CPU Cores    : {os.cpu_count() or 1}")

# Disk space
disk_root = "/content" if os.path.isdir("/content") else os.getcwd()
usage = shutil.disk_usage(disk_root)
print(f"Disk ({disk_root}): {usage.free / (1024**3):.1f} GB free / {usage.total / (1024**3):.1f} GB total")

# RAM
try:
    with open("/proc/meminfo", "r", encoding="utf-8") as fh:
        mem = {}
        for line in fh:
            parts = line.split()
            if len(parts) >= 2:
                mem[parts[0].rstrip(":")] = int(parts[1]) / (1024**2)
    print(f"System RAM   : {mem.get('MemAvailable', 0.0):.1f} GB available / {mem.get('MemTotal', 0.0):.1f} GB total")
except OSError:
    pass

# NVIDIA GPU check via nvidia-smi
nvidia_smi = shutil.which("nvidia-smi")
gpu_detected = False
if nvidia_smi:
    res = subprocess.run(
        [nvidia_smi, "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    if res.returncode == 0 and res.stdout.strip():
        gpu_detected = True
        print(f"NVIDIA GPU   : {res.stdout.strip()}")
    else:
        print(f"⚠️ nvidia-smi returned exit {res.returncode}: {(res.stderr or '').strip()}")

if not gpu_detected:
    print("⚠️ No NVIDIA GPU detected — PeakCut will run in CPU int8 fallback mode.")
    print("   Tip: In Google Colab, choose Runtime -> Change runtime type -> T4 GPU.")

# Check preinstalled PyTorch / CUDA status without modifying packages yet
try:
    import torch
    print(f"Preinstalled PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | CUDA build: {torch.version.cuda}")
except ImportError:
    print("Preinstalled PyTorch: not found (will be checked in Cell 4)")

if usage.free < 3 * (1024**3):
    raise RuntimeError(
        f"Less than 3 GB free disk space on {disk_root} ({usage.free / (1024**3):.2f} GB). "
        "Free disk space or start a fresh Colab runtime before continuing."
    )
print("✅ Cell 1 complete.")

## Cell 2 — Clone or Safely Update PeakCut Repository

In [ ]:
# @title 2. Clone or Safely Update PeakCut Repository
REPO_URL = "https://github.com/shreeharsh-patil/PeakCut.git"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/PeakCut"  # @param {type:"string"}
ALLOW_DISCARD_LOCAL_CHANGES = False  # @param {type:"boolean"}

import os
import subprocess
import sys

def _run(cmd, cwd=None):
    print(f"$ {' '.join(cmd)}")
    res = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout.strip())
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr.strip())
        raise RuntimeError(f"Command failed (exit {res.returncode}): {' '.join(cmd)}")
    return res

if not os.path.isdir(REPO_DIR):
    os.makedirs(os.path.dirname(REPO_DIR) or ".", exist_ok=True)
    _run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR])
else:
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git repository. Remove it or choose a different REPO_DIR."
        )
    status = subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True, capture_output=True, check=True
    ).stdout.strip()
    if status:
        if not ALLOW_DISCARD_LOCAL_CHANGES:
            print("⚠️ Local uncommitted changes detected in", REPO_DIR)
            print(status)
            print("ℹ️ Skipping git update to protect your local changes (set ALLOW_DISCARD_LOCAL_CHANGES=True to reset).")
        else:
            print("⚠️ Discarding local changes because ALLOW_DISCARD_LOCAL_CHANGES=True...")
            _run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
            _run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
            _run(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)
    else:
        _run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
        _run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
        _run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"✅ Repository ready at {REPO_DIR} (branch={REPO_BRANCH}, commit={commit})")

## Cell 3 — System Dependencies (FFmpeg, Fonts & Media Libraries)

In [ ]:
# @title 3. Install & Verify System Dependencies (FFmpeg, FFprobe, Fonts, OpenCV libs)
import os
import shutil
import subprocess
import sys

def _run_checked(cmd):
    print(f"$ {' '.join(cmd)}")
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout[-2000:] if res.stdout else "")
        print(res.stderr[-2000:] if res.stderr else "")
        raise RuntimeError(f"System installation failed (exit {res.returncode}): {' '.join(cmd)}")
    return res

if sys.platform.startswith("linux") and shutil.which("apt-get"):
    env = os.environ.copy()
    env["DEBIAN_FRONTEND"] = "noninteractive"
    print("📦 Updating apt package index and installing system dependencies...")
    subprocess.run(["apt-get", "update", "-qq"], check=True, env=env)
    packages = [
        "ffmpeg",
        "libsm6",
        "libxext6",
        "libgl1",
        "fonts-freefont-ttf",
        "fonts-dejavu-core",
        "fonts-noto-core",
        "fontconfig",
        "zstd",
        "curl",
        "nodejs",
    ]
    res = subprocess.run(["apt-get", "install", "-y", "-qq"] + packages, env=env, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stderr[-2000:] if res.stderr else "")
        raise RuntimeError(f"apt-get install failed with exit code {res.returncode}")
    if shutil.which("fc-cache"):
        subprocess.run(["fc-cache", "-f"], check=False)

# Strict verification: FFmpeg and FFprobe MUST be installed and runnable
ffmpeg_ver = _run_checked(["ffmpeg", "-version"]).stdout.splitlines()[0]
ffprobe_ver = _run_checked(["ffprobe", "-version"]).stdout.splitlines()[0]
print(f"✅ FFmpeg  : {ffmpeg_ver}")
print(f"✅ FFprobe : {ffprobe_ver}")

# Verify bundled subtitle fonts exist
bundled_fonts = [f for f in os.listdir("fonts") if f.lower().endswith((".ttf", ".otf"))] if os.path.isdir("fonts") else []
print(f"✅ Bundled subtitle fonts ({len(bundled_fonts)}): {', '.join(sorted(bundled_fonts)) or 'system fonts'}")

## Cell 4 — Python Dependencies (Preserving Colab Preinstalled PyTorch/CUDA)

In [ ]:
# @title 4. Install Python Dependencies Safely & Verify Imports
import importlib
import os
import shutil
import subprocess
import sys

def _pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"] + args
    print(f"$ {' '.join(cmd)}")
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout[-3000:] if res.stdout else "")
        print(res.stderr[-3000:] if res.stderr else "")
        raise RuntimeError(f"pip install failed (exit {res.returncode}): {' '.join(cmd)}")
    return res

# 1. Inspect preinstalled PyTorch/CUDA before touching GPU packages
has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode == 0

try:
    import torch
    import torchvision  # noqa: F401
    cuda_ok = torch.cuda.is_available()
    if has_gpu and not cuda_ok:
        print(f"⚠️ Preinstalled PyTorch ({torch.__version__}) has CUDA=False on a GPU runtime; keeping/checking compatibility.")
    else:
        print(f"✅ Preserving preinstalled PyTorch {torch.__version__} (CUDA={cuda_ok})")
except ImportError:
    print("📦 PyTorch not preinstalled — installing PyTorch...")
    _pip_install(["torch", "torchvision"])

# 2. Install Colab-safe dependencies (omits pinned torch/numpy to prevent ABI breakage)
req_file = "requirements-colab.txt" if os.path.isfile("requirements-colab.txt") else "requirements.txt"
_pip_install(["-r", req_file])

# 3. Always keep yt-dlp fresh for YouTube compatibility
_pip_install(["-U", "yt-dlp"])

# 4. Install PeakCut package editable (--no-deps so preinstalled PyTorch/CUDA is untouched)
_pip_install(["--no-deps", "-e", "."])

# 5. Install global `peakcut` CLI launcher (works from any directory)
import colab_runner
colab_runner.install_cli_launcher()

# 6. Strict import & version verification
required_modules = [
    ("torch", "PyTorch"),
    ("numpy", "NumPy"),
    ("cv2", "OpenCV"),
    ("faster_whisper", "Faster-Whisper"),
    ("yt_dlp", "yt-dlp"),
    ("fastapi", "FastAPI"),
    ("uvicorn", "Uvicorn"),
    ("PIL", "Pillow"),
]
print("\n🔍 Verifying core Python packages:")
failed = []
for mod_name, label in required_modules:
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, "__version__", None)
        if not ver and mod_name == "yt_dlp":
            ver = getattr(getattr(mod, "version", None), "__version__", "ok")
        print(f"   ✅ {label:16s}: {ver or 'ok'}")
    except Exception as exc:
        failed.append(f"{label} ({mod_name}): {exc}")

if failed:
    raise RuntimeError("Critical Python dependencies failed to import:\n  - " + "\n  - ".join(failed))

import torch
print(f"\n✅ Cell 4 complete. CUDA ready: {torch.cuda.is_available()}")

## Cell 5 — Google Drive Storage & Optional Secrets

In [ ]:
# @title 5. Mount Google Drive (Persistent Outputs) & Load Optional Colab Secrets
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

import os
import colab_runner

# Load optional Colab Secrets ONLY if user configured them (never required for local AI!)
try:
    from google.colab import userdata  # type: ignore
    for secret_name in ("GEMINI_API_KEY", "OPENAI_API_KEY", "YOUTUBE_ACCESS_TOKEN"):
        if not os.environ.get(secret_name):
            try:
                val = userdata.get(secret_name)
                if val and val.strip():
                    os.environ[secret_name] = val.strip()
                    print(f"🔑 Loaded optional Colab Secret: {secret_name}")
            except Exception:
                pass
except ImportError:
    pass

storage = colab_runner.mount_google_drive(mount_drive=USE_GOOGLE_DRIVE)
os.environ["OUTPUT_DIR"] = storage["work_dir"]
os.environ["UPLOAD_DIR"] = os.path.join(storage["work_dir"], "uploads")
os.makedirs(os.environ["UPLOAD_DIR"], exist_ok=True)
os.environ["PEAKCUT_DRIVE_SYNC_DIR"] = storage["output_root"]
os.environ["OLLAMA_MODELS"] = storage["models_dir"]

print("\n📁 Storage Configuration:")
print(f"   Google Drive mounted : {storage['drive_mounted']}")
print(f"   Fast SSD staging dir : {storage['work_dir']}")
print(f"   Saved Shorts folder  : {storage['output_root']}")
print(f"   Ollama models cache  : {storage['models_dir']}")

## Cell 6 — Local AI Initialization (Ollama + Gemma or Explicit Heuristic Fallback)

In [ ]:
# @title 6. Initialize Local AI Engine & Safe T4 Concurrency Profile
LOCAL_MODEL = "gemma2:2b"  # @param ["gemma2:2b", "llama3.2:3b", "qwen2.5:3b", "phi3:mini", "gemma2:9b"]
WHISPER_MODEL = "base"  # @param ["base", "small", "tiny", "medium"]
RENDER_PROFILE = "balanced"  # @param ["balanced", "low_resource", "high_quality"]
RUN_TEST_INFERENCE = True  # @param {type:"boolean"}

import os
import colab_runner

env_status = colab_runner.setup_colab_environment(
    model_name=LOCAL_MODEL,
    mount_drive=bool(os.environ.get("PEAKCUT_DRIVE_SYNC_DIR", "").startswith("/content/drive")),
    whisper_model=WHISPER_MODEL,
    force_local=True,
    render_profile=RENDER_PROFILE,
    preserve_external_keys=True,
    run_test_inference=RUN_TEST_INFERENCE,
)

print("\n📋 Local AI & Hardware Summary:")
print(f"   Hardware       : {env_status['device']} (CUDA={env_status['has_cuda']})")
print(f"   AI Mode        : {env_status['ai_mode']}")
print(f"   Ollama Ready   : {env_status['ollama_ready']} (model={env_status['model']})")
print(f"   Output Folder  : {env_status['output_root']}")
if not env_status["ollama_ready"]:
    print("   ℹ️ Ollama did not pass full readiness checks — PeakCut will use its heuristic clip selector.")
    print(f"      Detail: {env_status['ai_status'].get('error')}")

## Cell 7 — Start PeakCut Backend & Verify Local Health

In [ ]:
# @title 7. Start FastAPI/Uvicorn Backend FIRST & Verify All Core Endpoints
BACKEND_PORT = 8000  # @param {type:"integer"}
ENABLE_TUNNEL_AUTH = True  # @param {type:"boolean"}

import colab_runner

backend_info = colab_runner.start_backend_server(
    host="127.0.0.1",
    port=int(BACKEND_PORT),
    require_auth=bool(ENABLE_TUNNEL_AUTH),
    startup_timeout=60.0,
)

print(f"\n✅ Backend PID  : {backend_info['pid']}")
print(f"✅ Local URL    : {backend_info['local_url']}")
print(f"🔒 Token Auth   : {'ENABLED' if backend_info['auth_enabled'] else 'DISABLED'}")
print(f"📜 Backend Log  : {backend_info['backend_log']}")

## Cell 8 — Open Secure PeakCut Studio Dashboard

In [ ]:
# @title 8. Expose Dashboard (Authenticated Cloudflare Tunnel or Private Colab Proxy)
ACCESS_MODE = "cloudflare"  # @param ["cloudflare", "colab_proxy"]

import colab_runner

port = int(globals().get("BACKEND_PORT", 8000))
require_auth = bool(globals().get("ENABLE_TUNNEL_AUTH", True))

dashboard_url = colab_runner.start_dashboard_access(
    port=port,
    host="127.0.0.1",
    use_tunnel=(ACCESS_MODE == "cloudflare"),
    access_mode=ACCESS_MODE,
    require_auth=require_auth,
)
print("✅ Dashboard access active. Notebook remains unblocked so you can run Cells 9–11 anytime.")

## Cell 9 — Process a Video (YouTube URL or Local Upload)

In [ ]:
# @title 9. Generate Vertical Shorts with Animated Subtitles (URL or Local File)
VIDEO_SOURCE = ""  # @param {type:"string"}
UPLOAD_LOCAL_VIDEO = False  # @param {type:"boolean"}
CLIPS_PER_VIDEO = 3  # @param {type:"integer"}
OUTPUT_FORMAT = "vertical"  # @param ["vertical", "horizontal", "square", "auto"]
SUBTITLE_PRESET = "karaoke"  # @param ["karaoke", "minimal", "bold", "none"]
CLIP_MIN_SECONDS = 15  # @param {type:"integer"}
CLIP_MAX_SECONDS = 55  # @param {type:"integer"}
RANGE_START_SECONDS = 0  # @param {type:"number"}
RANGE_END_SECONDS = 0  # @param {type:"number"}
CLEANUP_TEMP_FILES = True  # @param {type:"boolean"}

import os
import colab_runner

inputs = []
if UPLOAD_LOCAL_VIDEO:
    try:
        from google.colab import files as colab_files  # type: ignore
        print("📤 Select a local video file (.mp4 / .mov / .mkv) to upload...")
        uploaded = colab_files.upload()
        upload_dir = os.environ.get("UPLOAD_DIR") or "/content/PeakCut_Work/uploads"
        os.makedirs(upload_dir, exist_ok=True)
        for fname, data in uploaded.items():
            dest = os.path.join(upload_dir, os.path.basename(fname))
            with open(dest, "wb") as fh:
                fh.write(data)
            inputs.append(dest)
            print(f"✅ Uploaded {fname} -> {dest} ({len(data)/(1024*1024):.1f} MB)")
    except ImportError:
        raise RuntimeError("UPLOAD_LOCAL_VIDEO requires Google Colab; pass a local file path in VIDEO_SOURCE instead.")

if VIDEO_SOURCE.strip():
    inputs.extend(colab_runner.parse_url_list(VIDEO_SOURCE))

if not inputs:
    raise ValueError("Provide a YouTube URL or local file path in VIDEO_SOURCE, or enable UPLOAD_LOCAL_VIDEO=True.")

pipeline_results = colab_runner.run_shorts_pipeline(
    urls=inputs,
    clips_per_video=int(CLIPS_PER_VIDEO),
    format=OUTPUT_FORMAT,
    clip_min_seconds=float(CLIP_MIN_SECONDS),
    clip_max_seconds=float(CLIP_MAX_SECONDS),
    subtitle_preset=SUBTITLE_PRESET,
    range_start=float(RANGE_START_SECONDS) if RANGE_START_SECONDS > 0 else None,
    range_end=float(RANGE_END_SECONDS) if RANGE_END_SECONDS > 0 else None,
    cleanup_temp=bool(CLEANUP_TEMP_FILES),
)

if pipeline_results.get("failed") and not pipeline_results.get("successful"):
    first_err = pipeline_results["failed"][0].get("error", "Unknown error")
    raise RuntimeError(f"Video processing failed: {first_err}")

## Cell 10 — Preview Generated Clips, Download & CLI Examples

In [ ]:
# @title 10. Inspect Saved Clips, Preview Latest Short & Download
DOWNLOAD_LATEST_CLIP = False  # @param {type:"boolean"}

import base64
import glob
import os
import subprocess

out_root = (
    os.environ.get("PEAKCUT_DRIVE_SYNC_DIR")
    or os.environ.get("OUTPUT_DIR")
    or "/content/PeakCut_Output"
)

mp4_files = sorted(
    [
        p for p in glob.glob(os.path.join(out_root, "**", "*.mp4"), recursive=True)
        if not os.path.basename(p).startswith("temp_")
    ],
    key=os.path.getmtime,
    reverse=True,
)

print(f"📁 Output directory: {out_root}")
print(f"🎬 Verified MP4 clips found: {len(mp4_files)}")

for idx, path in enumerate(mp4_files[:10], 1):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    dur_str = "?"
    try:
        out = subprocess.check_output(
            [
                "ffprobe", "-v", "error", "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1", path,
            ],
            text=True, timeout=10,
        ).strip()
        if out:
            dur_str = f"{float(out):.1f}s"
    except Exception:
        pass
    print(f"  {idx}. {os.path.relpath(path, out_root)}  ({size_mb:.2f} MB, {dur_str})")

if mp4_files:
    latest = mp4_files[0]
    size_bytes = os.path.getsize(latest)
    if size_bytes <= 40 * 1024 * 1024:
        try:
            from IPython.display import HTML, display
            b64 = base64.b64encode(open(latest, "rb").read()).decode("ascii")
            display(HTML(
                f"<h4>🎬 Latest Clip Preview: {os.path.basename(latest)}</h4>"
                f"<video width='320' controls style='border-radius:12px;border:1px solid #3f3f46;'>"
                f"<source src='data:video/mp4;base64,{b64}' type='video/mp4'></video>"
            ))
        except Exception as e:
            print(f"ℹ️ Inline preview skipped: {e}")
    else:
        print(f"ℹ️ Latest clip is {size_bytes/(1024*1024):.1f} MB (>40 MB); open directly from Google Drive.")

    if DOWNLOAD_LATEST_CLIP:
        try:
            from google.colab import files as colab_files  # type: ignore
            colab_files.download(latest)
        except ImportError:
            print(f"ℹ️ Download helper is Colab-only. File path: {latest}")
else:
    print("ℹ️ No generated clips yet. Run Cell 9 or submit a job in the PeakCut Studio Dashboard.")

print("\n💻 Verified CLI Commands (usable in any terminal or notebook cell):")
print('   peakcut status')
print('   peakcut diagnose')
print('   peakcut clip "https://www.youtube.com/watch?v=VIDEO_ID" --clips 3 --format vertical --subtitles karaoke')
print('   peakcut serve --background')
print('   peakcut stop')

## Cell 11 — Diagnose PeakCut, Inspect Logs & Safe Service Stop

In [ ]:
# @title 11. Diagnose PeakCut (14-Point Automated Health Report & Logs)
TEST_LOCAL_AI_INFERENCE = False  # @param {type:"boolean"}
SHOW_SERVICE_LOGS = True  # @param {type:"boolean"}
STOP_ALL_SERVICES = False  # @param {type:"boolean"}

from pathlib import Path
import colab_runner

port = int(globals().get("BACKEND_PORT", 8000))

if STOP_ALL_SERVICES:
    stopped = colab_runner.stop_managed_services(("cloudflare", "backend", "ollama"))
    print(f"🛑 Safely stopped PeakCut-managed services: {stopped}")
else:
    report = colab_runner.diagnose_peakcut(port=port, run_inference_check=bool(TEST_LOCAL_AI_INFERENCE))
    if SHOW_SERVICE_LOGS:
        log_dir = Path(report["log_dir"])
        for name in ("backend.log", "cloudflare.log", "ollama.log"):
            log_file = log_dir / name
            if log_file.exists():
                print(f"\n--- Last 20 lines of {log_file} ---")
                print(colab_runner._scrub_token_text(colab_runner._tail_log(log_file, 20)))